In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import tensorflow as tf
import tensorflow.keras.layers as L
import random, warnings

from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

seed_everything()

2026-03-24 06:53:36.635440: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774335216.843364      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774335216.903099      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774335217.413117      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774335217.413155      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774335217.413158      55 computation_placer.cc:177] computation placer alr

In [8]:
DATASET_PATH = "/kaggle/input/datasets/shubham2703/five-crop-diseases-dataset/Crop Diseases Dataset/Crop Diseases/Crop___Disease"

IMAGE_SIZE = 128
BATCH_SIZE = 16

In [9]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_gen = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

NUM_CLASSES = train_gen.num_classes

Found 10661 images belonging to 5 classes.
Found 2663 images belonging to 5 classes.


In [16]:
def conv_embedding(x, filters, kernel_size, strides):
    x = L.Conv2D(filters, kernel_size, strides=strides, padding="same")(x)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)

    # Convert to sequence (tokens)
    x = L.Reshape((-1, filters))(x)
    return x

In [19]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = L.Dense(units, activation=tf.nn.gelu)(x)
        x = L.Dropout(dropout_rate)(x)
    return x

In [21]:
def transformer_block(x, projection_dim, num_heads):
    x1 = L.LayerNormalization(epsilon=1e-6)(x)
    
    attn = L.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim
    )(x1, x1)
    
    x2 = L.Add()([attn, x])
    
    x3 = L.LayerNormalization(epsilon=1e-6)(x2)
    x3 = mlp(x3, [projection_dim*2, projection_dim], 0.1)
    
    return L.Add()([x3, x2])

In [22]:
def cvt_model():
    inputs = L.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

    # 🔥 Convolutional embedding
    x = conv_embedding(inputs, filters=64, kernel_size=7, strides=4)

    # 🔥 3 Transformer Blocks
    x = transformer_block(x, projection_dim=64, num_heads=4)
    x = transformer_block(x, projection_dim=64, num_heads=4)
    x = transformer_block(x, projection_dim=64, num_heads=4)

    # Classification head
    x = L.LayerNormalization(epsilon=1e-6)(x)
    x = L.GlobalAveragePooling1D()(x)
    x = L.Dropout(0.3)(x)

    outputs = L.Dense(NUM_CLASSES, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)

In [5]:
model = cvt_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

NameError: name 'cvt_model' is not defined

In [25]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

Epoch 1/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2290s 3s/step - accuracy: 0.8045 - loss: 0.8076 - val_accuracy: 0.9001 - val_loss: 0.6374
Epoch 2/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2237s 3s/step - accuracy: 0.9534 - loss: 0.5292 - val_accuracy: 0.8055 - val_loss: 0.8991
Epoch 3/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2315s 3s/step - accuracy: 0.9742 - loss: 0.4835 - val_accuracy: 0.9594 - val_loss: 0.5088
Epoch 4/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2183s 3s/step - accuracy: 0.9781 - loss: 0.4675 - val_accuracy: 0.9204 - val_loss: 0.6097
Epoch 5/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2213s 3s/step - accuracy: 0.9821 - loss: 0.4533 - val_accuracy: 0.8137 - val_loss: 0.8005
Epoch 6/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2313s 3s/step - accuracy: 0.9834 - loss: 0.4480 - val_accuracy: 0.9448 - val_loss: 0.5382
Epoch 7/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2226s 3s/step - accuracy: 0.9847 - loss: 0.4424 - val_accuracy: 0.6331 - val_loss: 1.0950
Epoch 8/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 2227s 3s/step - accuracy: 0.9853 - loss: 0.4379 - 